# Phase 1: Public Online Data Collection (Model H — Telugu)

This notebook downloads and saves the **non-manual** (public/online) datasets for the Telugu language model.

### Data Sources

1. **IndicCorpV2** (`ai4bharat/IndicCorpV2`, split `tel_Telu`) — Massive monolingual web crawl corpus (~500M+ words)
2. **CulturaX Telugu** (`LightFury9/CulturaX_500ksamples_8splits_te`) — Filtered web crawl text (~180M words)
3. **Telugu Novels** (`eswardivi/telugu_dataset`) — Novel chapters and literary prose (~18M words)
4. **Telugu OCR** (`harsha-desaraju/telugu-ocr-text`) — Scanned Telugu books via OCR (~13M words)

---

## Summary

| File | Docs (Lines) | Est. Words | Size | Source |
| :--- | :---: | :---: | :---: | :--- |
| `telugu_indiccorp.jsonl` | ~16.1M | ~569M | ~12.6 GB | ai4bharat/IndicCorpV2 (tel_Telu) |
| `telugu_culturax.jsonl` | ~500K | ~180M | ~3.5 GB | CulturaX Telugu subset |
| `telugu_novels.jsonl` | ~47K | ~18M | ~374 MB | eswardivi/telugu_dataset |
| `telugu_ocr.jsonl` | ~50K | ~13M | ~252 MB | harsha-desaraju/telugu-ocr-text |
| **TOTAL** | **~16.7M** | **~780M** | **~16.7 GB** | |

## 1. Setup

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

RAW_DOWNLOAD_DIR = Path("../datasets/raw/downloaded/")
RAW_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {RAW_DOWNLOAD_DIR.resolve()}")

## 2. Download IndicCorpV2 — Telugu (500M+ words)

Streams the `tel_Telu` split of `ai4bharat/IndicCorpV2` and saves ~16M+ documents to JSONL.
Each document is a short sentence/paragraph averaging ~17 words.

**Runtime**: ~10–15 minutes (depends on network speed).  
**Output**: `telugu_indiccorp.jsonl` (~12.6 GB)

In [ ]:
out_path = RAW_DOWNLOAD_DIR / "telugu_indiccorp.jsonl"

# Target: enough docs to get 500M+ words (~17 words/doc avg -> ~30M docs)
LIMIT = 30_000_000

if out_path.exists() and out_path.stat().st_size > 1_000_000_000:  # >1GB means already downloaded
    print(f"Already exists: {out_path.name} ({out_path.stat().st_size / 1e9:.2f} GB). Skipping.")
    print("Delete the file to re-download.")
else:
    print(f"Loading IndicCorpV2 Telugu (tel_Telu) — streaming up to {LIMIT/1e6:.0f}M docs...")
    ds = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split="tel_Telu", streaming=True)

    total_docs = 0
    total_words = 0

    with open(out_path, "w", encoding="utf-8") as f:
        for item in tqdm(ds, total=LIMIT, desc="Downloading IndicCorpV2 (Telugu)"):
            if total_docs >= LIMIT:
                break
            text = item["text"].strip()
            if text:
                f.write(json.dumps({"text": text, "source": "indiccorpv2_telugu"}, ensure_ascii=False) + "\n")
                total_docs += 1
                total_words += len(text.split())
            if total_docs % 5_000_000 == 0:
                print(f"  {total_docs/1e6:.0f}M docs, {total_words/1e6:.1f}M words, {out_path.stat().st_size/1e9:.2f} GB")

    print(f"\nDone. Saved {total_docs:,} docs ({total_words/1e6:.1f}M words) → {out_path.name}")
    print(f"File size: {out_path.stat().st_size / 1e9:.2f} GB")

## 3. Download CulturaX Telugu (~180M words)

Downloads the CulturaX Telugu subset — filtered web crawl text from mC4/OSCAR pipelines.

**Output**: `telugu_culturax.jsonl` (~3.5 GB)

In [ ]:
out_path = RAW_DOWNLOAD_DIR / "telugu_culturax.jsonl"

if out_path.exists() and out_path.stat().st_size > 100_000_000:  # >100MB
    print(f"Already exists: {out_path.name} ({out_path.stat().st_size / 1e9:.2f} GB). Skipping.")
else:
    print("Loading CulturaX Telugu subset...")
    ds = load_dataset("LightFury9/CulturaX_500ksamples_8splits_te", split="train", streaming=True)

    total_docs = 0
    LIMIT = 500_000

    with open(out_path, "w", encoding="utf-8") as f:
        for item in tqdm(ds, total=LIMIT, desc="Downloading CulturaX Telugu"):
            if total_docs >= LIMIT:
                break
            text = item.get("text", "").strip()
            if text:
                f.write(json.dumps({"text": text, "source": "culturax_telugu"}, ensure_ascii=False) + "\n")
                total_docs += 1

    print(f"Done. Saved {total_docs:,} docs → {out_path.name}")

## 4. Download Telugu Novels (~18M words)

Downloads Telugu literary prose — novel chapters, dialogues, and narrative text.

**Output**: `telugu_novels.jsonl` (~374 MB)

In [ ]:
out_path = RAW_DOWNLOAD_DIR / "telugu_novels.jsonl"

if out_path.exists() and out_path.stat().st_size > 10_000_000:  # >10MB
    print(f"Already exists: {out_path.name} ({out_path.stat().st_size / 1e6:.0f} MB). Skipping.")
else:
    print("Loading Telugu Novels dataset...")
    ds = load_dataset("eswardivi/telugu_dataset", split="train")

    total_docs = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for item in tqdm(ds, desc="Saving Telugu Novels"):
            text = item.get("text", "").strip()
            if text:
                f.write(json.dumps({"text": text, "source": "telugu_novels"}, ensure_ascii=False) + "\n")
                total_docs += 1

    print(f"Done. Saved {total_docs:,} docs → {out_path.name}")

## 5. Download Telugu OCR Text (~13M words)

Downloads OCR-extracted text from scanned Telugu books.

**Output**: `telugu_ocr.jsonl` (~252 MB)

In [ ]:
out_path = RAW_DOWNLOAD_DIR / "telugu_ocr.jsonl"

if out_path.exists() and out_path.stat().st_size > 10_000_000:  # >10MB
    print(f"Already exists: {out_path.name} ({out_path.stat().st_size / 1e6:.0f} MB). Skipping.")
else:
    print("Loading Telugu OCR dataset...")
    ds = load_dataset("harsha-desaraju/telugu-ocr-text", split="train")

    total_docs = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for item in tqdm(ds, desc="Saving Telugu OCR"):
            text = item.get("text", "").strip()
            if text:
                f.write(json.dumps({"text": text, "source": "telugu_ocr"}, ensure_ascii=False) + "\n")
                total_docs += 1

    print(f"Done. Saved {total_docs:,} docs → {out_path.name}")

## 6. Dataset Audit

Compute word and character counts for all downloaded JSONL files.

In [ ]:
import re

telugu_re = re.compile(r"[\u0C00-\u0C7F]")

all_files = sorted(RAW_DOWNLOAD_DIR.glob("*.jsonl"))
print(f"{'File':<40} {'Lines':>12} {'Words':>14} {'Chars':>16} {'Size (MB)':>10}")
print("-" * 96)
grand_lines = grand_words = grand_chars = 0

for fpath in all_files:
    lines = words = chars = 0
    with open(fpath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                text = json.loads(line).get("text", "")
            except Exception:
                text = line
            lines += 1
            words += len(text.split())
            chars += len(text)
    grand_lines += lines
    grand_words += words
    grand_chars += chars
    size_mb = fpath.stat().st_size / 1024 / 1024
    print(f"{fpath.name:<40} {lines:>12,} {words:>14,} {chars:>16,} {size_mb:>10,.1f}")

print("-" * 96)
print(f"{'TOTAL':<40} {grand_lines:>12,} {grand_words:>14,} {grand_chars:>16,}")
print(f"\nTotal words: {grand_words/1e6:.1f}M")

## 7. Sample Inspection

In [ ]:
import random

def inspect_samples(filepath, n=5):
    """Print random samples from a JSONL file."""
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return
    samples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 50000:
                break
            if line.strip():
                samples.append(line.strip())
    chosen = random.sample(samples, min(n, len(samples)))
    print(f"\n{'='*80}")
    print(f"Samples from: {filepath.name}")
    print(f"{'='*80}")
    for i, line in enumerate(chosen, 1):
        try:
            text = json.loads(line).get("text", "")[:300]
        except:
            text = line[:300]
        print(f"\n[{i}] {text}")
        print("-" * 40)

for f in sorted(RAW_DOWNLOAD_DIR.glob("*.jsonl")):
    inspect_samples(f, n=3)